# 🧠 Transfer Learning Implementation with VGG16

> **Module:** 03 — Deep Learning with Keras and TensorFlow  
> **Topic:** Transfer learning, feature extraction, fine-tuning, optimizer comparison  
> **Base model:** VGG16 (ImageNet weights, 138M parameters)

---

## 📋 Overview

In this notebook I implement **transfer learning** — taking VGG16 trained on ImageNet and repurposing its learned representations for a new binary classification task.

| Part | Step | Description |
|---|---|---|
| Part 1 | Load VGG16 | Import pretrained base, freeze all layers |
| Part 2 | Build classifier | Add Flatten + Dense head on top |
| Part 3 | Feature extraction | Train only the new head |
| Part 4 | Fine-tuning | Unfreeze last 4 VGG16 layers, retrain |
| Exercises | Validation, optimizers, test eval | Solved below |

## 🧩 Theory

### Why transfer learning?

Training a deep CNN from scratch requires millions of labelled images and days of GPU compute. VGG16, trained on 1.2 million ImageNet images, already knows how to detect edges, textures, shapes, and object parts. I **borrow** those representations and adapt them with far less data.

**Telecom / RF analogy 📡:** Transfer learning is like reusing a calibrated RF front-end (LNA, filter bank, ADC chain) from a high-budget platform and connecting only a new baseband block tuned to your specific standard. The front-end cost is amortised across many deployments.

### Two-phase strategy

| Phase | Layers frozen | What trains | Purpose |
|---|---|---|---|
| **Feature extraction** | All VGG16 layers | New Dense head only | Fast convergence; pretrained features unchanged |
| **Fine-tuning** | All except last N | Last N VGG16 layers + Dense head | Adapt high-level features to new domain |

### VGG16 architecture (no top)

```
Input (224×224×3)
  Block 1: Conv(64)×2 → MaxPool   → 112×112×64
  Block 2: Conv(128)×2 → MaxPool  → 56×56×128
  Block 3: Conv(256)×3 → MaxPool  → 28×28×256
  Block 4: Conv(512)×3 → MaxPool  → 14×14×512
  Block 5: Conv(512)×3 → MaxPool  → 7×7×512
  [Top excluded: FC(4096)×2 → Softmax(1000)]
```

### Binary cross-entropy loss

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}\left[y_i \log\hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\right]$$

### Why unfreeze only the last layers?

Earlier layers detect generic low-level features (edges, colours) — highly transferable. Later layers encode task-specific high-level patterns — more likely to need adaptation:
$$\text{Layer specificity} \propto \text{depth}$$

## ⚙️ Part 0 — Setup

In [ ]:
!pip install tensorflow==2.16.2 matplotlib==3.9.1 --quiet

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential, clone_model
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f"TensorFlow: {tf.__version__}")

## 🏗️ Part 1 — Load VGG16 (Feature Extraction Base)

I load VGG16 with `include_top=False` (no classifier head) and freeze all layers.

| Parameter | Value | Meaning |
|---|---|---|
| `weights='imagenet'` | ImageNet pretrained | 138M parameters trained on 1.2M images |
| `include_top=False` | No classifier head | Excludes FC(4096)×2 + Softmax(1000) |
| `input_shape=(224,224,3)` | Fixed input size | VGG16 designed for 224×224 RGB |

In [ ]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze ALL base model layers
for layer in base_model.layers:
    layer.trainable = False

trainable     = sum(tf.size(v).numpy() for v in base_model.trainable_weights)
non_trainable = sum(tf.size(v).numpy() for v in base_model.non_trainable_weights)
print(f"Trainable params:     {trainable:,}")
print(f"Non-trainable params: {non_trainable:,}")
print(f"Output shape (no top): {base_model.output_shape}")

With all layers frozen, trainable params = 0. The 7×7×512 output feeds my custom classifier head.

## 🎯 Part 2 — Build & Compile the Model

I attach a new classifier head on top of the frozen base:
```
VGG16 base (frozen) → (7, 7, 512)
  Flatten           → (25088,)      [7×7×512]
  Dense(256, ReLU)  → (256,)
  Dense(1, sigmoid) → (1,)          [binary probability]
```

Loss: $\mathcal{L} = -[y \log\hat{y} + (1-y)\log(1-\hat{y})]$

In [ ]:
model = Sequential([
    base_model,
    Flatten(),
    Dense(256, activation='relu'),
    Dense(1,   activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 📥 Part 3 — Prepare Data & Train (Feature Extraction)

I generate two synthetic classes:
- **class_a:** white images (255) — 'signal present'
- **class_b:** black images (0) — 'signal absent'

`flow_from_directory` reads images from a folder where each subfolder = one class label, resizes to 224×224, and rescales pixels to $[0, 1]$.

In [ ]:
os.makedirs('sample_data/class_a', exist_ok=True)
os.makedirs('sample_data/class_b', exist_ok=True)

for i in range(10):
    img = Image.fromarray(np.ones((224, 224, 3), dtype=np.uint8) * 255)
    img.save(f'sample_data/class_a/img_{i}.jpg')
    img = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
    img.save(f'sample_data/class_b/img_{i}.jpg')

print("✅ Sample data created: 10 images per class")

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_directory(
    'sample_data',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

print(f"Class indices: {train_generator.class_indices}")
print(f"Total images:  {train_generator.samples}")

if train_generator.samples > 0:
    print("\n🏋️ Phase 1 — Feature Extraction (frozen VGG16)")
    history_phase1 = model.fit(train_generator, epochs=10, verbose=1)

## 🔓 Part 4 — Fine-Tuning

I unfreeze the **last 4 VGG16 layers** (Block 5 conv layers + pool) and recompile. This adapts the highest-level features to my specific task while keeping the lower-level general features intact.

> **Always recompile after changing `trainable` flags** — Keras registers gradients at compile time.

In [ ]:
# Unfreeze last 4 layers of VGG16
for layer in base_model.layers[-4:]:
    layer.trainable = True

print("Trainable layers after unfreezing:")
for layer in base_model.layers:
    if layer.trainable:
        print(f"  ✅ {layer.name}")

# Recompile to register new trainable params with the optimiser
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("\n🔬 Phase 2 — Fine-Tuning (last 4 VGG16 layers unfrozen)")
history_phase2 = model.fit(train_generator, epochs=10, verbose=1)

---

## 🔢 Exercises — Solved

### Exercise 1 — Visualise Training & Validation Loss

In [ ]:
train_datagen_val = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_generator_val = train_datagen_val.flow_from_directory(
    'sample_data', target_size=(224, 224), batch_size=32,
    class_mode='binary', subset='training'
)
validation_generator = train_datagen_val.flow_from_directory(
    'sample_data', target_size=(224, 224), batch_size=32,
    class_mode='binary', subset='validation'
)

history_val = model.fit(
    train_generator_val, epochs=10,
    validation_data=validation_generator, verbose=1
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_val.history['loss'],         label='Train Loss',     color='steelblue')
axes[0].plot(history_val.history['val_loss'],     label='Val Loss',       color='tomato', linestyle='--')
axes[0].set_title('📉 Training vs Validation Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history_val.history['accuracy'],     label='Train Accuracy', color='steelblue')
axes[1].plot(history_val.history['val_accuracy'], label='Val Accuracy',   color='tomato', linestyle='--')
axes[1].set_title('📈 Training vs Validation Accuracy')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('✅ Exercise 1 — Loss & Accuracy Curves', fontsize=13)
plt.tight_layout(); plt.show()

### Exercise 2 — Optimizer Comparison: SGD vs RMSprop vs Adam

| Optimizer | Update rule | Key property |
|---|---|---|
| **SGD** | $\theta \leftarrow \theta - \alpha \nabla \mathcal{L}$ | Simple, stable, slower convergence |
| **RMSprop** | $\theta \leftarrow \theta - \frac{\alpha}{\sqrt{v + \epsilon}} \nabla \mathcal{L}$ | Adaptive LR; good for non-stationary problems |
| **Adam** | $\theta \leftarrow \theta - \frac{\alpha \hat{m}}{\sqrt{\hat{v}} + \epsilon}$ | Momentum + RMSprop; fastest convergence |

In [ ]:
def reset_model(reference_model):
    cloned = clone_model(reference_model)
    cloned.set_weights(reference_model.get_weights())
    return cloned

initial_model = reset_model(model)
results = {}

for opt_name in ['sgd', 'rmsprop', 'adam']:
    print(f"\n⚙️ Training with {opt_name.upper()}...")
    m = reset_model(initial_model)
    m.compile(optimizer=opt_name, loss='binary_crossentropy', metrics=['accuracy'])
    hist = m.fit(train_generator_val, epochs=10, validation_data=validation_generator, verbose=0)
    results[opt_name] = hist.history
    print(f"   Final val accuracy: {hist.history['val_accuracy'][-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'sgd': 'coral', 'rmsprop': 'mediumseagreen', 'adam': 'steelblue'}
for opt_name, hist in results.items():
    axes[0].plot(hist['accuracy'],     label=f'{opt_name.upper()} train', color=colors[opt_name])
    axes[0].plot(hist['val_accuracy'], label=f'{opt_name.upper()} val',   color=colors[opt_name], linestyle='--')
    axes[1].plot(hist['loss'],     label=f'{opt_name.upper()} train', color=colors[opt_name])
    axes[1].plot(hist['val_loss'], label=f'{opt_name.upper()} val',   color=colors[opt_name], linestyle='--')
axes[0].set_title('📊 Accuracy — All Optimisers'); axes[0].set_xlabel('Epoch'); axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
axes[1].set_title('📉 Loss — All Optimisers'); axes[1].set_xlabel('Epoch'); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
plt.suptitle('✅ Exercise 2 — SGD vs RMSprop vs Adam', fontsize=13)
plt.tight_layout(); plt.show()

print('\n📊 Final validation accuracy:')
for opt_name, hist in results.items():
    print(f'  {opt_name.upper():10s}: {hist["val_accuracy"][-1]:.4f}')

### Exercise 3 — Evaluate on Test Set

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    'sample_data', target_size=(224, 224),
    batch_size=32, class_mode='binary', shuffle=False
)

test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)
print(f"\n🎯 Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"   Test Loss:     {test_loss:.4f}")

---

## 📊 Summary

| Step | What I did | Key API |
|---|---|---|
| Load base | VGG16 without top, ImageNet weights | `VGG16(weights='imagenet', include_top=False)` |
| Freeze base | All layers non-trainable | `layer.trainable = False` |
| Build head | Flatten → Dense(256, ReLU) → Dense(1, sigmoid) | `Sequential([base_model, ...])` |
| Phase 1 | Only Dense head trains | `model.fit(generator, epochs=10)` |
| Fine-tune | Unfreeze last 4 VGG16 layers | `layer.trainable = True` + `model.compile()` |
| Phase 2 | Last 4 conv layers + head update | `model.fit(generator, epochs=10)` |
| Evaluate | Test loss and accuracy | `model.evaluate(test_generator)` |

### Key rules
- Always **recompile** after changing `trainable` flags
- Use a **lower LR** during fine-tuning to avoid destroying pretrained weights
- Fine-tune only **last few layers** — early layers have generic, transferable features
- Test generator must use **training-set normalisation statistics**
- Adam converges faster; SGD can generalise better with large datasets

---

## 🧪 Sandbox

In [ ]:
# SANDBOX 1: Inspect VGG16 layer names and trainability
print(f"{'Idx':>3}  {'Name':<25}  {'Type':<25}  {'Trainable'}")
print('-' * 75)
for i, layer in enumerate(base_model.layers):
    print(f"{i:>3}  {layer.name:<25}  {type(layer).__name__:<25}  {layer.trainable}")

In [ ]:
# SANDBOX 2: Fine-tune Block 5 with a lower LR (best practice)
# Telecom analogy: like a PLL with reduced loop bandwidth during lock-in
block5_layers = ['block5_conv1', 'block5_conv2', 'block5_conv3', 'block5_pool']
for layer in base_model.layers:
    layer.trainable = layer.name in block5_layers

model_ft = reset_model(model)
model_ft.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy', metrics=['accuracy']
)
print("Fine-tuning Block 5 with LR=1e-5")
history_ft = model_ft.fit(train_generator_val, epochs=5, validation_data=validation_generator, verbose=1)

plt.figure(figsize=(8, 4))
plt.plot(history_ft.history['loss'],     label='Train loss')
plt.plot(history_ft.history['val_loss'], label='Val loss', linestyle='--')
plt.title('🧪 Sandbox 2 — Fine-Tuning Block 5 with Low LR (1e-5)')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# SANDBOX 3: Visualise Block 1 Conv 1 activation maps
activation_model = tf.keras.Model(
    inputs=base_model.input,
    outputs=base_model.get_layer('block1_conv1').output
)
sample_img = np.ones((1, 224, 224, 3), dtype=np.float32)
activations = activation_model.predict(sample_img, verbose=0)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for i, ax in enumerate(axes.flat):
    ax.imshow(activations[0, :, :, i], cmap='viridis')
    ax.set_title(f'Filter {i}', fontsize=8)
    ax.axis('off')
plt.suptitle('🧪 Sandbox 3 — Block 1 Conv 1 Activation Maps (16/64 filters)', fontsize=12)
plt.tight_layout(); plt.show()